In [1]:
import scipy.io
import numpy as np
from rssp.sam_clustering import build_rssp_tree, print_tree

data   = scipy.io.loadmat('data/indian_pines/Indian_pines_corrected.mat')['indian_pines_corrected']
labels = scipy.io.loadmat('data/indian_pines/Indian_pines_gt.mat')['indian_pines_gt']

# (C, H, W) format
data_chw = data.transpose(2, 0, 1).astype(np.float32)

tree, sam_matrix, means = build_rssp_tree(
    data_chw, labels, num_classes=17, depth_mode='auto'
)

print_tree(tree)

Building RSSP tree for 16 classes
Pixel counts: {1: 46, 2: 1428, 3: 830, 4: 237, 5: 483, 6: 730, 7: 28, 8: 478, 9: 20, 10: 972, 11: 2455, 12: 593, 13: 205, 14: 1265, 15: 386, 16: 93}
Node (depth 0): classes [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
  ├── LEFT:
    Node (depth 1): classes [11, 10, 6, 8, 4, 13, 1]
      ├── LEFT:
        Node (depth 2): classes [11, 1]
      └── RIGHT:
        Node (depth 2): classes [10, 6, 8, 4, 13]
          ├── LEFT:
            Node (depth 3): classes [10, 4]
          └── RIGHT:
            Node (depth 3): classes [6, 8, 13]
              ├── LEFT:
                Node (depth 4): classes [6]
              └── RIGHT:
                Node (depth 4): classes [8, 13]
  └── RIGHT:
    Node (depth 1): classes [2, 14, 3, 12, 5, 15, 16, 7, 9]
      ├── LEFT:
        Node (depth 2): classes [2, 12, 5, 7, 9]
          ├── LEFT:
            Node (depth 3): classes [2]
          └── RIGHT:
            Node (depth 3): classes [12, 5, 7, 9]
       

In [3]:
import scipy.io
import numpy as np
import torch
from rssp.rssp_trainer import NodeDataset

data   = scipy.io.loadmat('data/indian_pines/Indian_pines_corrected.mat')['indian_pines_corrected']
labels = scipy.io.loadmat('data/indian_pines/Indian_pines_gt.mat')['indian_pines_gt']

data_chw = torch.tensor(data.transpose(2,0,1).astype(np.float32))
labels_t = torch.tensor(labels.astype(np.int64))

# Simulate root node with all classes
all_coords = np.argwhere(labels > 0)
ds = NodeDataset(data_chw, labels_t, 
                 node_classes=list(range(1,17)),
                 split_coords=all_coords)

d, l = ds[0]
print(d.shape)          # (200, 145, 145)
print(l.shape)          # (145, 145)
print(l.max())          # 16
print(l.min())          # 0
print(ds.global_to_local)  # {1:1, 2:2, ... 16:16} for root

torch.Size([200, 145, 145])
torch.Size([145, 145])
tensor(16)
tensor(0)
{1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16}


In [4]:
data   = scipy.io.loadmat('data/pavia/PaviaU.mat')['paviaU']
labels = scipy.io.loadmat('data/pavia/PaviaU_gt.mat')['paviaU_gt']

print(f"Data shape:   {data.shape}")    # should be (610, 340, 103)
print(f"Labels shape: {labels.shape}") # should be (610, 340)
print(f"Num classes:  {labels.max()}") # should be 9

# Tree
data_chw = data.transpose(2,0,1).astype(np.float32)
tree, sam_matrix, means = build_rssp_tree(
    data_chw, labels,
    num_classes=int(labels.max())+1,
    depth_mode='auto'
)
print_tree(tree)

Data shape:   (610, 340, 103)
Labels shape: (610, 340)
Num classes:  9
Building RSSP tree for 9 classes
Pixel counts: {1: 6631, 2: 18649, 3: 2099, 4: 3064, 5: 1345, 6: 5029, 7: 1330, 8: 3682, 9: 947}
Node (depth 0): classes [1, 2, 3, 4, 5, 6, 7, 8, 9]
  ├── LEFT:
    Node (depth 1): classes [2, 5, 7]
      ├── LEFT:
        Node (depth 2): classes [2]
      └── RIGHT:
        Node (depth 2): classes [5, 7]
  └── RIGHT:
    Node (depth 1): classes [1, 6, 8, 4, 3, 9]
      ├── LEFT:
        Node (depth 2): classes [1, 4, 9]
          ├── LEFT:
            Node (depth 3): classes [1]
          └── RIGHT:
            Node (depth 3): classes [4, 9]
      └── RIGHT:
        Node (depth 2): classes [6, 8, 3]
          ├── LEFT:
            Node (depth 3): classes [6]
          └── RIGHT:
            Node (depth 3): classes [8, 3]


In [5]:
data_t   = torch.tensor(data_chw)
labels_t = torch.tensor(labels.astype(np.int64))
all_coords = np.argwhere(labels > 0)

ds = NodeDataset(data_t, labels_t,
                 node_classes=list(range(1, int(labels.max())+1)),
                 split_coords=all_coords)

d, l = ds[0]
print(d.shape)   # (103, 610, 340)
print(l.shape)   # (610, 340)
print(l.max())   # 9
print(l.min())   # 0

torch.Size([103, 610, 340])
torch.Size([610, 340])
tensor(9)
tensor(0)


In [6]:
# After training, check what root alone predicts
from rssp.rssp_inference import predict_node
import scipy.io, torch

data   = torch.tensor(
    scipy.io.loadmat('data/indian_pines/Indian_pines_corrected.mat')
    ['indian_pines_corrected'].transpose(2,0,1).astype('float32')
).unsqueeze(0)

with open('rssp_models.pkl', 'rb') as f:
    import pickle
    saved = pickle.load(f)

trained_models = saved['trained_models']
root_pred, root_conf = predict_node(
    trained_models['root'], data, 'cuda', voting='weighted'
)
print(f"Root unique predictions: {set(root_pred.flatten().tolist())}")
print(f"Mean confidence: {root_conf.mean():.3f}")
print(f"Min confidence:  {root_conf.min():.3f}")

Root unique predictions: {0, 3, 5, 6, 7, 14}
Mean confidence: 0.139
Min confidence:  0.069
